# MSPE 49600 — Project 4: Fluent and Gaussian Process Modeling

**Software:** VS Code (Jupyter notebook) + ANSYS Student 2026 R1.  
**Source data:** `airfoil_results.csv` (10 completed Fluent design points).  
**Units:** rotation in degrees, drag and lift in N, variance in N².

Run these cells **in order** after the full ANSYS sweep has finished. The notebook does **not** invent CFD results. The signed lift convention is **+y upward**, so downforce is `-lift_N`. Confirm those directions in your Fluent force reports before running the notebook.

## Step 8 — Load and validate the ten CFD runs

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel, RBF

INPUT_CSV = Path("airfoil_results.csv")
OUTPUT_CSV = Path("Aramthun_pr04_cfd.csv")

if not INPUT_CSV.exists():
    raise FileNotFoundError("Run the ten-angle Fluent sweep first; airfoil_results.csv is missing.")

required = ["angle_deg", "drag_N", "lift_N"]
df = pd.read_csv(INPUT_CSV)
if not set(required).issubset(df.columns):
    raise ValueError(f"Expected columns: {required}; found: {df.columns.tolist()}")
df = df[required].copy()
for col in required:
    df[col] = pd.to_numeric(df[col], errors="raise")
if len(df) != 10:
    raise ValueError(f"Expected exactly 10 CFD rows; found {len(df)}.")
if df.isna().any().any() or not np.isfinite(df.to_numpy(dtype=float)).all():
    raise ValueError("Missing or nonfinite angle/force values detected.")
if df["angle_deg"].duplicated().any():
    raise ValueError("Duplicate CFD angles detected.")
print("Validated 10 finite, unique CFD samples.")
print("Verify in Fluent: drag +x, lift +y, both forces in N.")
display(df)

## Step 9 — Downforce and ratio

**Threshold choice:** Set the minimum usable downforce to **5% of the largest observed positive downforce** (a positive, data-scaled cutoff). This is a screening heuristic, not a vehicle requirement: it keeps the ratio from appearing favorable when its denominator is nearly zero. Ratios below the threshold remain blank (`NaN`), but both forces remain in the dataset. If the engineering team gives an absolute minimum downforce, replace this heuristic with that physical requirement.

In [ ]:
df = df.sort_values("angle_deg").reset_index(drop=True)
df["downforce_N"] = -df["lift_N"]
positive_downforce = df.loc[df["downforce_N"] > 0, "downforce_N"]
if positive_downforce.empty:
    raise ValueError(
        "No positive downforce found. Check Fluent lift sign/angle orientation before analyzing ratios."
    )
MIN_DOWNFORCE_N = 0.05 * float(positive_downforce.max())
print(f"Minimum downforce for ratio = {MIN_DOWNFORCE_N:.6g} N")

df["drag_to_downforce"] = np.nan
eligible = df["downforce_N"] >= MIN_DOWNFORCE_N
df.loc[eligible, "drag_to_downforce"] = (
    df.loc[eligible, "drag_N"] / df.loc[eligible, "downforce_N"]
)
export_cols = ["angle_deg", "drag_N", "lift_N", "drag_to_downforce"]
df[export_cols].to_csv(OUTPUT_CSV, index=False)
print(f"Saved {OUTPUT_CSV.resolve()}")
display(df[export_cols])

## Steps 10–11 — Held-out split and separate drag/lift GPs

In [ ]:
test_rows = [2, 5, 8]  # 3rd, 6th, and 9th rows AFTER sorting
train = df.drop(index=test_rows).copy()
test = df.iloc[test_rows].copy()
X_train = train[["angle_deg"]]
X_test = test[["angle_deg"]]

def make_gp():
    return GaussianProcessRegressor(
        kernel=ConstantKernel() * RBF(), normalize_y=True
    )

gp_drag = make_gp()
gp_lift = make_gp()
gp_drag.fit(X_train, train["drag_N"])
gp_lift.fit(X_train, train["lift_N"])

for name, model in [("Drag", gp_drag), ("Lift", gp_lift)]:
    print(f"{name} fitted kernel: {model.kernel_}")
    print(f"  Kernel variance (normalized scale): {model.kernel_.k1.constant_value:.8g}")
    print(f"  RBF length scale (degrees): {model.kernel_.k2.length_scale:.8g}")
print(f"Training runs: {len(train)} | Test runs: {len(test)}")

## Steps 12–13 — Predict the three test runs, residuals, and RMSE

In [ ]:
test_results = pd.DataFrame({
    "angle_deg": test["angle_deg"].to_numpy(),
    "observed_drag_N": test["drag_N"].to_numpy(),
    "predicted_drag_N": gp_drag.predict(X_test),
    "observed_lift_N": test["lift_N"].to_numpy(),
    "predicted_lift_N": gp_lift.predict(X_test),
})
test_results["drag_residual_N"] = (
    test_results["observed_drag_N"] - test_results["predicted_drag_N"]
)
test_results["lift_residual_N"] = (
    test_results["observed_lift_N"] - test_results["predicted_lift_N"]
)
rmse_drag_N = float(np.sqrt(np.mean(test_results["drag_residual_N"]**2)))
rmse_lift_N = float(np.sqrt(np.mean(test_results["lift_residual_N"]**2)))
print(f"Drag test RMSE = {rmse_drag_N:.6g} N")
print(f"Lift test RMSE = {rmse_lift_N:.6g} N")
display(test_results)

## Step 14 — Observed-versus-predicted test plots

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, force in zip(axes, ["drag", "lift"]):
    obs = test_results[f"observed_{force}_N"].to_numpy()
    pred = test_results[f"predicted_{force}_N"].to_numpy()
    lo, hi = min(obs.min(), pred.min()), max(obs.max(), pred.max())
    pad = max((hi - lo) * 0.08, 1e-8)
    ax.scatter(obs, pred, label="Held-out CFD runs")
    ax.plot([lo-pad, hi+pad], [lo-pad, hi+pad], "--", label="Ideal: y = x")
    ax.set(xlabel=f"Observed {force} (N)", ylabel=f"Predicted {force} (N)",
           title=f"{force.title()}: observed vs. predicted")
    ax.grid(alpha=0.3)
    ax.legend()
plt.tight_layout()
plt.show()

for force in ["drag", "lift"]:
    i = test_results[f"{force}_residual_N"].abs().idxmax()
    row = test_results.loc[i]
    print(f"Largest {force} absolute test error: {abs(row[f'{force}_residual_N']):.6g} N "
          f"at {row['angle_deg']:.6g}°")
print("Interpretation: large held-out errors make predicted performance near that angle less reliable.")

## Step 15 — Refit both models on all ten CFD runs

In [ ]:
X_all = df[["angle_deg"]]
gp_drag_full = make_gp().fit(X_all, df["drag_N"])
gp_lift_full = make_gp().fit(X_all, df["lift_N"])
for name, model in [("Drag", gp_drag_full), ("Lift", gp_lift_full)]:
    print(f"Final {name} kernel: {model.kernel_}")
    print(f"  Kernel variance: {model.kernel_.k1.constant_value:.8g}")
    print(f"  Length scale: {model.kernel_.k2.length_scale:.8g} deg")

grid_angles = np.linspace(df["angle_deg"].min(), df["angle_deg"].max(), 401)
X_grid = pd.DataFrame({"angle_deg": grid_angles})
drag_mu, drag_std = gp_drag_full.predict(X_grid, return_std=True)
lift_mu, lift_std = gp_lift_full.predict(X_grid, return_std=True)
drag_var, lift_var = drag_std**2, lift_std**2

## Step 16 — Predicted forces, 95% pointwise bands, and variance

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharex="col")
for j, (name, measured, mu, std, var) in enumerate([
    ("Drag", df["drag_N"], drag_mu, drag_std, drag_var),
    ("Lift", df["lift_N"], lift_mu, lift_std, lift_var),
]):
    axes[0, j].scatter(df["angle_deg"], measured, label="CFD", zorder=3)
    axes[0, j].plot(grid_angles, mu, label="GP mean")
    axes[0, j].fill_between(grid_angles, mu-1.96*std, mu+1.96*std,
                            alpha=0.22, label="Mean ± 1.96 std")
    axes[0, j].set(ylabel=f"{name} (N)", title=f"{name}: mean and pointwise band")
    axes[0, j].legend()
    axes[1, j].plot(grid_angles, var)
    axes[1, j].set(xlabel="CAD angle (deg)", ylabel="Predictive variance (N²)",
                   title=f"{name}: GP predictive variance")
    for ax in (axes[0, j], axes[1, j]):
        ax.grid(alpha=0.3)
    idx = int(np.argmax(var))
    print(f"Highest {name.lower()} GP variance at {grid_angles[idx]:.6g}°, "
          f"variance = {var[idx]:.6g} N²")
plt.tight_layout()
plt.show()
print("GP variance quantifies model prediction uncertainty under the fitted kernel; "
      "it does not directly measure mesh error, convergence error, or turbulence-model bias.")

## Steps 17–18 — Select a predicted candidate and assess its evidence

In [ ]:
predicted_downforce = -lift_mu
grid_ratio = np.full_like(grid_angles, np.nan, dtype=float)
grid_eligible = predicted_downforce >= MIN_DOWNFORCE_N
grid_ratio[grid_eligible] = drag_mu[grid_eligible] / predicted_downforce[grid_eligible]

if not np.isfinite(grid_ratio).any():
    raise ValueError("No predicted grid angle meets the chosen positive downforce threshold.")
if not df["drag_to_downforce"].notna().any():
    raise ValueError("No sampled CFD angle meets the minimum downforce threshold.")

best_idx = int(np.nanargmin(grid_ratio))
best_sample = df.loc[df["drag_to_downforce"].idxmin()]
recommended_angle_deg = float(grid_angles[best_idx])  # Candidate for verification, not a final setup.
at_boundary = best_idx == 0 or best_idx == len(grid_angles) - 1

print(f"Best GP-grid ratio: {grid_ratio[best_idx]:.6g} at {recommended_angle_deg:.6g}°")
print(f"  Predicted drag: {drag_mu[best_idx]:.6g} N")
print(f"  Predicted downforce: {predicted_downforce[best_idx]:.6g} N")
print(f"  At sampled boundary: {at_boundary}")
print(f"  GP drag variance nearby: {drag_var[best_idx]:.6g} N²")
print(f"  GP lift variance nearby: {lift_var[best_idx]:.6g} N²")
print(f"Lowest observed sampled ratio: {best_sample['drag_to_downforce']:.6g} "
      f"at {best_sample['angle_deg']:.6g}°")
print(f"  Observed drag: {best_sample['drag_N']:.6g} N")
print(f"  Observed downforce: {best_sample['downforce_N']:.6g} N")
print(f"Held-out RMSE: drag {rmse_drag_N:.6g} N; lift {rmse_lift_N:.6g} N")
print("Caution: the lowest D/downforce ratio does not mean the car gets enough downforce.")

## Step 19 — Fluent verification and engineering discussion

**Proposed additional CFD run:** Set the Workbench CAD angle to `recommended_angle_deg`, regenerate mesh if needed, confirm boundary conditions and convergence, and record new drag and signed lift. Compute the new ratio only if the verified downforce meets `MIN_DOWNFORCE_N`. Compare verified forces and ratio with the GP means at that angle; compare force differences with the held-out test RMSE and the local GP standard deviations.

**Separate limitations to discuss in the submission:**

- Mesh size and near-wall/wake resolution can bias drag and lift even when a GP is confident.
- Residual and force-history convergence must be checked for **every** Fluent run.
- Turbulence-model choice can bias predicted separation, wake and stall behavior.
- A 2D isolated wing section cannot capture 3D wingtip vortices, endplates, finite-span effects, or full-car interactions.
- The sampled-domain boundary and held-out force errors affect confidence in the recommended angle.

Use the values printed in Steps 13, 16, and 17–18 to write your final recommendation; do not claim that the additional Fluent verification has already been completed.

In [ ]:
display(Markdown(
    f"**Proposed verification angle:** {recommended_angle_deg:.4f}°  \n"
    f"**Predicted drag:** {drag_mu[best_idx]:.4f} N  \n"
    f"**Predicted lift:** {lift_mu[best_idx]:.4f} N  \n"
    f"**Predicted downforce:** {predicted_downforce[best_idx]:.4f} N  \n"
    f"**Predicted drag/downforce:** {grid_ratio[best_idx]:.5f}  \n"
    f"**Held-out drag RMSE:** {rmse_drag_N:.4f} N  \n"
    f"**Held-out lift RMSE:** {rmse_lift_N:.4f} N"
))